## 📝 Instrucciones ##

**Análisis de sentimientos**

Los modelos Naive Bayes son muy útiles cuando queremos analizar sentimientos, clasificar textos en tópicos o recomendaciones, ya que las características de estos desafíos cumplen muy bien con los supuestos teóricos y metodológicos del modelo.

En este proyecto practicarás con un conjunto de datos para crear un clasificador de reseñas de la tienda de Google Play.

### Paso 1: Carga del conjunto de datos ###

El conjunto de datos se puede encontrar en esta carpeta de proyecto bajo el nombre playstore_reviews.csv. Puedes cargarlo en el código directamente desde el sigiente enlace:

https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv

O descargarlo y añadirlo a mano en tu repositorio. En este conjunto de datos encontrarás las siguientes variables:

    -package_name. Nombre de la aplicación móvil (categórico)

    -review. Comentario sobre la aplicación móvil (categórico)

    -polarity. Variable de clase (0 o 1), siendo 0 un comentario negativo y 1, positivo (categórico numérico)

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score 
from sklearn.ensemble import RandomForestClassifier

In [11]:
df = pd.read_csv('../data/raw/playstore_reviews.csv')
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


### Paso 2: Estudio de variables y su contenido ###

En este caso, tenemos solo 3 variables: 2 predictoras y una etiqueta dicotómica. De las dos predictoras, realmente solo nos interesa la parte del comentario, ya que el hecho de clasificar un comentario en positivo o negativo dependerá de su contenido, no de la aplicación de la que se haya escrito. Por lo tanto, la variable package_name habría que eliminarla.

Cuando trabajamos con textos como en este caso, no tiene sentido hacer un EDA, el proceso es diferente, ya que la única variable que nos interesa es la que contiene el texto. En otros casos en los que el texto formase parte de un conjunto complejo con otras variables predictoras numéricas y el objetivo de predicción sea distinto, entonces tiene sentido aplicar un EDA.

Sin embargo, no podemos trabajar con texto plano, antes hay que procesarlo. Este proceso consta de varios pasos:

Eliminar espacios y convertir a minúsculas el texto:

**df["column"] = df["column"].str.strip().str.lower()**

Dividir el conjunto de datos en train y test: X_train, X_test, y_train, y_test

Transformar el texto en una matriz de recuento de palabras. Esta es una forma de obtener características numéricas a partir del texto. Para ello, utilizamos el conjunto de train para entrenar el transformador y la aplicamos en test:

**vec_model = CountVectorizer(stop_words = "english")**

**X_train = vec_model.fit_transform(X_train).toarray()**

**X_test = vec_model.transform(X_test).toarray()**

Una vez hayamos terminado tendremos listas las predictoras para entrenar el modelo.

In [12]:
#Eliminamos la clumna package_name que no se va a usar
df = df.drop(columns=['package_name'])
df.head()

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0


In [14]:
#Limpiar el texto: eliminar espacios y convetir a minúsculas
df['review'] = df['review'].str.strip().str.lower()
df.head()

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0


In [ ]:
#Verificar que el texto esté limpio
print("Valores nulos:", df['review'].isnull().sum())
print("Valores vacíos:", (df['review'] == '').sum())

Valores nulos: 0
Valores vacíos: 0


In [18]:
X = df['review']
y = df['polarity']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
vec_model = CountVectorizer(stop_words='english')
X_train_vect = vec_model.fit_transform(X_train).toarray()
X_test_vect = vec_model.transform(X_test).toarray()

print("X_train_vect shape:", X_train_vect.shape)
print("X_test_vect shape:", X_test_vect.shape)

X_train_vect shape: (712, 3310)
X_test_vect shape: (179, 3310)


### Paso 3: Construye un Naive Bayes ###

Comienza a resolver el problema implementando un modelo del que tendrás que elegir cuál de las tres implementaciones utilizar: GaussianNB, MultinomialNB o BernoulliNB, según lo que hemos estudiado en el módulo. Prueba ahora a entrenarlo con las dos otras implementaciones y confirma si el modelo que has elegido es el adecuado.

**MultinomialNB**

In [26]:
mnb = MultinomialNB()
mnb.fit(X_train_vect, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [ ]:
#Predicciones
y_pred_mnb = mnb.predict(X_test_vect)

print("Accuracy MultinomialNB:", accuracy_score(y_test, y_pred_mnb))
print("Classification Report:\n", classification_report(y_test, y_pred_mnb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_mnb))

Accuracy MultinomialNB: 0.8156424581005587
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87       126
           1       0.73      0.60      0.66        53

    accuracy                           0.82       179
   macro avg       0.79      0.75      0.77       179
weighted avg       0.81      0.82      0.81       179

Confusion Matrix:
 [[114  12]
 [ 21  32]]


**GaussianNB**

In [33]:
gnb = GaussianNB()
gnb.fit(X_train_vect, y_train)
y_pred_gnb = gnb.predict(X_test_vect)
print("Accuracy GaussianNB:", accuracy_score(y_test, y_pred_gnb))
print("Classification Report:\n", classification_report(y_test, y_pred_gnb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gnb))

Accuracy GaussianNB: 0.8044692737430168
Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.88      0.86       126
           1       0.69      0.62      0.65        53

    accuracy                           0.80       179
   macro avg       0.77      0.75      0.76       179
weighted avg       0.80      0.80      0.80       179

Confusion Matrix:
 [[111  15]
 [ 20  33]]


**Bernoulli**

In [34]:
bnb = BernoulliNB()
bnb.fit(X_train_vect, y_train)
y_pred_bnb = bnb.predict(X_test_vect)
print("Accuracy BernoulliNB:", accuracy_score(y_test, y_pred_bnb))
print("Classification Report:\n", classification_report(y_test, y_pred_bnb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bnb))

Accuracy BernoulliNB: 0.770949720670391
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.93      0.85       126
           1       0.70      0.40      0.51        53

    accuracy                           0.77       179
   macro avg       0.74      0.66      0.68       179
weighted avg       0.76      0.77      0.75       179

Confusion Matrix:
 [[117   9]
 [ 32  21]]


**Comparación de modelos Naive Bayes**

| Métrica         | MultinomialNB | GaussianNB | BernoulliNB |
|-----------------|---------------|------------|-------------|
| Accuracy        | 0.816         | 0.804      | 0.771       |
| Precision Clase 0 | 0.84        | 0.85       | 0.79        |
| Precision Clase 1 | 0.73        | 0.69       | 0.70        |
| Recall Clase 0    | 0.90        | 0.88       | 0.93        |
| Recall Clase 1    | 0.60        | 0.62       | 0.40        |
| F1-score Clase 0  | 0.87        | 0.86       | 0.85        |
| F1-score Clase 1  | 0.66        | 0.65       | 0.51        |

1. MultinomialNB

_Mejor accuracy general (0.816)._

Teóricamente este modelo asume que los datos son conteos o frecuencias de palabras, lo cual coincide bien con el método CountVectorizes que fue el que se usó para vectorizar el texto. Los resultados parecen confirmar esta conveniencia respecto a los otros 2 métodos.

2. GaussianNB

Similar a MultinomialNB, pero ligeramente menor precisión y recall en la clase positiva.

Menos adecuado porque asume que las características siguen distribución normal, lo cual no se cumple en datos de conteo de palabras.

3. BernoulliNB

Mejor en detectar la clase negativa (recall 0 = 0.93) pero muy débil en la clase positiva (recall 1 = 0.40).

Funciona mejor con datos binarios (presencia/ausencia de palabra), pero pierde información del conteo de palabras, por eso falla con comentarios positivos.


**Conclusión: MultinomialNB es el modelo más equilibrado para este dataset ya que detecta correctamente la mayoría de los comentarios negativos y positivos. Es consistente con la naturaleza del texto y el vectorizado de conteos. GaussianNB y BernoulliNB son alternativas, pero no superan el desempeño general de MultinomialNB.**

### Paso 4: Optimiza el modelo anterior ###

Después de entrenar el modelo en sus tres implementaciones, elige la mejor opción y trata de optimizar sus resultados con un random forest, si es posible

In [37]:
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest.fit(X_train_vect, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [39]:
y_pred_random_forest = random_forest.predict(X_test_vect)
accuracy_random_forest = accuracy_score(y_test, y_pred_random_forest)
print("Accuracy Random Forest:", accuracy_random_forest)
print("Classification Report:\n", classification_report(y_test, y_pred_random_forest))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_random_forest))

Accuracy Random Forest: 0.7988826815642458
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.83      0.85       126
           1       0.64      0.74      0.68        53

    accuracy                           0.80       179
   macro avg       0.76      0.78      0.77       179
weighted avg       0.81      0.80      0.80       179

Confusion Matrix:
 [[104  22]
 [ 14  39]]


**Comparación de modelos de clasificación (vertical)**

| Métrica           | MultinomialNB | GaussianNB | BernoulliNB | Random Forest |
|------------------|---------------|------------|-------------|---------------|
| Accuracy          | 0.816         | 0.804      | 0.771       | 0.799         |
| Precision Clase 0 | 0.84          | 0.85       | 0.79        | 0.88          |
| Precision Clase 1 | 0.73          | 0.69       | 0.70        | 0.64          |
| Recall Clase 0    | 0.90          | 0.88       | 0.93        | 0.83          |
| Recall Clase 1    | 0.60          | 0.62       | 0.40        | 0.74          |
| F1-score Clase 0  | 0.87          | 0.86       | 0.85        | 0.85          |
| F1-score Clase 1  | 0.66          | 0.65       | 0.51        | 0.68          |



MultinomialNB sigue siendo el modelo más equilibrado. Random Forest detecta mejor la clase positiva (recall = 0.74) pero empeora la clase negatia (recall = 0.83). No mejora el desempeño general y es más costoso computacionalmente. 

### Paso 5: Guarda el modelo ###

Almacena el modelo en la carpeta correspondiente.

In [42]:
import joblib
joblib.dump(mnb, "../models/playstore_reviews/multinomial_nb_model.pkl") 


['../models/playstore_reviews/multinomial_nb_model.pkl']

### Paso 6: Explora otras alternativas ###

¿Qué otros modelos de los que hemos estudiado podrías utilizar para intentar superar los resultados de un Naive Bayes? Arguméntalo y entrena el modelo.

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

In [48]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_vect, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [49]:
y_pred_log_reg = log_reg.predict(X_test_vect)
accuracy_log_reg = accuracy_score(y_test, y_pred_log_reg)
print("Accuracy Logistic Regression:", accuracy_log_reg)
print("Classification Report:\n", classification_report(y_test, y_pred_log_reg))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log_reg))

Accuracy Logistic Regression: 0.8324022346368715
Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.84      0.88       126
           1       0.68      0.81      0.74        53

    accuracy                           0.83       179
   macro avg       0.80      0.83      0.81       179
weighted avg       0.85      0.83      0.84       179

Confusion Matrix:
 [[106  20]
 [ 10  43]]


In [52]:
grad_boost = GradientBoostingClassifier(n_estimators=100, random_state=42)
grad_boost.fit(X_train_vect, y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [53]:
y_pred_grad_boost = grad_boost.predict(X_test_vect)
accuracy_grad_boost = accuracy_score(y_test, y_pred_grad_boost)
print("Accuracy Gradient Boosting:", accuracy_grad_boost)
print("Classification Report:\n", classification_report(y_test, y_pred_grad_boost))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_grad_boost))

Accuracy Gradient Boosting: 0.7318435754189944
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.83      0.81       126
           1       0.55      0.49      0.52        53

    accuracy                           0.73       179
   macro avg       0.67      0.66      0.67       179
weighted avg       0.72      0.73      0.73       179

Confusion Matrix:
 [[105  21]
 [ 27  26]]


| Modelo                  | Métrica           | Clase 0 (negativo) | Clase 1 (positivo) | Accuracy |
|-------------------------|-----------------|------------------|------------------|---------|
| MultinomialNB           | Precision       | 0.84             | 0.73             | 0.816   |
|                         | Recall          | 0.90             | 0.60             |         |
|                         | F1-score        | 0.87             | 0.66             |         |
| GaussianNB              | Precision       | 0.85             | 0.69             | 0.804   |
|                         | Recall          | 0.88             | 0.62             |         |
|                         | F1-score        | 0.86             | 0.65             |         |
| BernoulliNB             | Precision       | 0.79             | 0.70             | 0.771   |
|                         | Recall          | 0.93             | 0.40             |         |
|                         | F1-score        | 0.85             | 0.51             |         |
| Random Forest           | Precision       | 0.88             | 0.64             | 0.799   |
|                         | Recall          | 0.83             | 0.74             |         |
|                         | F1-score        | 0.85             | 0.68             |         |
| Logistic Regression     | Precision       | 0.91             | 0.68             | 0.832   |
|                         | Recall          | 0.84             | 0.81             |         |
|                         | F1-score        | 0.88             | 0.74             |         |
| Gradient Boosting       | Precision       | 0.80             | 0.55             | 0.732   |
|                         | Recall          | 0.83             | 0.49             |         |
|                         | F1-score        | 0.81             | 0.52             |         |

**Conclusión: Entre todos los métodos, Logistic Regression es la opción más equilibrada y efectiva, especialmente si nos interesa detectar correctamente tanto comentarios positivos como negativos.**